<a href="https://colab.research.google.com/github/gabrielcabanilla1/EXSU-500-Clinical-NLP-large-language-models-rct/blob/main/notebooks/03a_position_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03a – Position baseline (statistical baseline)
Predicts the role of a sentence from **where it sits in the abstract only**; the words are never read.
For each of 10 position bins (0 = first sentence, 9 = last), the label frequencies are counted on the
training split. The predicted label is the most frequent one in that bin, and the frequencies are used
as the class probabilities, so AUROC and AUPRC can be computed like for the other models.

This is the floor every other model must beat. Input: the files prepared by 02_data_preparation.
Only the training and validation splits are used here; the test set is evaluated once, in 06_final_test.

**Decision rule.** All reported labels, F1, sensitivity and specificity use the same rule as the other
models: the class with the highest probability (argmax).

In [1]:
# ===== CONFIGURATION =====
DATA_VERSION = 'PubMed_20k_RCT'
PROJECT_DIR = '/content/drive/MyDrive/exsu_project'
DATA_DIR = f'{PROJECT_DIR}/prepared_data'
CLASSES = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']

N_POSITION_BINS = 10   # same bins as 02_data_preparation, fixed before looking at validation results
SMOOTHING = 1          # add-one smoothing: no class gets probability 0 in a bin
N_BOOTSTRAP = 2000     # bootstrap resamples of whole abstracts for the confidence interval
SEED = 42

RUN_NAME = 'position_baseline'
MODEL_PATH = f'{PROJECT_DIR}/models/{RUN_NAME}_table.csv'
PREDICTIONS_PATH = f'{PROJECT_DIR}/predictions/dev_{RUN_NAME}.csv'
RESULTS_DIR = f'{PROJECT_DIR}/results'
print('Run:', RUN_NAME, '| position bins:', N_POSITION_BINS)

Run: position_baseline | position bins: 10


In [2]:
import os
import time
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
for folder in [os.path.dirname(MODEL_PATH), os.path.dirname(PREDICTIONS_PATH), RESULTS_DIR]:
    os.makedirs(folder, exist_ok=True)

def load_split(split):
    return pd.read_csv(f'{DATA_DIR}/{DATA_VERSION}_{split}.csv',
                       dtype={'abstract_id': str}, keep_default_na=False)

train, dev = load_split('train'), load_split('dev')
assert len(train) == 180040 and len(dev) == 30212, 'Unexpected size: re-run 02_data_preparation'
assert not set(train.abstract_id) & set(dev.abstract_id), 'Leakage: abstract in train and dev'
print(f'Train: {len(train):,} sentences | Dev: {len(dev):,} sentences')

Mounted at /content/drive
Train: 180,040 sentences | Dev: 30,212 sentences


## Fit: label frequencies per position bin (training split only)

In [3]:
def position_bin(df):
    """0 = first sentence of the abstract, N_POSITION_BINS - 1 = last sentence."""
    return np.minimum((df.position * N_POSITION_BINS).astype(int), N_POSITION_BINS - 1)

for df in (train, dev):
    df['bin'] = position_bin(df)
if N_POSITION_BINS == 10:   # must match the column made in 02_data_preparation
    assert (train.bin == train.position_bin).all() and (dev.bin == dev.position_bin).all()

start = time.time()
counts = (pd.crosstab(train.bin, train.label)
            .reindex(index=range(N_POSITION_BINS), columns=CLASSES, fill_value=0))
smoothed = counts + SMOOTHING
probability_table = smoothed.div(smoothed.sum(axis=1), axis=0)   # P(label | position bin)
fit_seconds = time.time() - start

rule = probability_table.idxmax(axis=1)   # most frequent label in each bin
probability_table.assign(predicted_label=rule).to_csv(MODEL_PATH)
print(f'Fitted in {fit_seconds:.3f} s on CPU. Saved to {MODEL_PATH}\n')
print('Training sentences per bin and label, with the label the rule predicts:')
display(counts.assign(predicted_label=rule))

Fitted in 0.056 s on CPU. Saved to /content/drive/MyDrive/exsu_project/models/position_baseline_table.csv

Training sentences per bin and label, with the label the rule predicts:


label,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS,predicted_label
bin,,,,,,
0,11283,9405,2261,27,17,BACKGROUND
1,4499,2886,8338,225,5,METHODS
2,1863,1193,13195,679,0,METHODS
3,520,258,13210,2072,6,METHODS
4,164,58,9736,4390,14,METHODS
5,59,22,7667,10920,44,RESULTS
6,20,9,3022,13623,263,RESULTS
7,28,4,1174,13526,1327,RESULTS
8,118,3,443,10082,6178,RESULTS


## Predict on the validation split

In [4]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

dev_probs = probability_table.loc[dev.bin].to_numpy()        # n x 5, columns in CLASSES order
dev_pred = np.array(CLASSES)[dev_probs.argmax(axis=1)]      # argmax decision rule

accuracy = accuracy_score(dev.label, dev_pred)
macro_f1 = f1_score(dev.label, dev_pred, labels=CLASSES, average='macro', zero_division=0)
print(f'Dev accuracy: {accuracy:.4f}')
print(f'Dev macro-F1: {macro_f1:.4f}\n')
print(classification_report(dev.label, dev_pred, labels=CLASSES, digits=4, zero_division=0))

never_predicted = [c for c in CLASSES if c not in set(dev_pred)]
print('Classes the rule never predicts:', never_predicted or 'none')

# Simplest possible reference: always predict the most frequent training label
majority = train.label.value_counts().idxmax()
majority_pred = np.full(len(dev), majority)
print(f'\nReference, always "{majority}": accuracy {accuracy_score(dev.label, majority_pred):.4f}, '
      f'macro-F1 {f1_score(dev.label, majority_pred, labels=CLASSES, average="macro", zero_division=0):.4f}')

Dev accuracy: 0.6887
Dev macro-F1: 0.5477

              precision    recall  f1-score   support

  BACKGROUND     0.4693    0.5274    0.4967      3449
   OBJECTIVE     0.0000    0.0000    0.0000      2376
     METHODS     0.7097    0.7559    0.7321      9964
     RESULTS     0.7119    0.8317    0.7672      9841
 CONCLUSIONS     0.7740    0.7139    0.7427      4582

    accuracy                         0.6887     30212
   macro avg     0.5330    0.5658    0.5477     30212
weighted avg     0.6369    0.6887    0.6607     30212

Classes the rule never predicts: ['OBJECTIVE']

Reference, always "METHODS": accuracy 0.3298, macro-F1 0.0992


In [5]:
counts_matrix = confusion_matrix(dev.label, dev_pred, labels=CLASSES)
confusion = pd.DataFrame(counts_matrix, index=pd.Index(CLASSES, name='Actual'),
                         columns=pd.Index(CLASSES, name='Predicted'))
display(confusion)

Predicted,BACKGROUND,OBJECTIVE,METHODS,RESULTS,CONCLUSIONS
Actual,,,,,
BACKGROUND,1819,0,1093,34,503
OBJECTIVE,1646,0,728,2,0
METHODS,401,0,7532,1974,57
RESULTS,7,0,1254,8185,395
CONCLUSIONS,3,0,6,1302,3271


## Per-class metrics (one class vs. the other four)
AUROC and AUPRC use the class probabilities. Sensitivity and specificity use the argmax rule, so they
match the F1 above. Note: this model gives only 10 distinct probability values per class (one per bin),
so its ROC and precision-recall curves have very few steps.

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score

rows = []
for i, c in enumerate(CLASSES):
    actual = dev.label.eq(c).to_numpy()
    predicted = dev_pred == c
    rows.append({
        'Class': c,
        'AUROC': roc_auc_score(actual, dev_probs[:, i]),
        'AUPRC (AP)': average_precision_score(actual, dev_probs[:, i]),
        'Sensitivity': predicted[actual].mean(),
        'Specificity': (~predicted[~actual]).mean(),
        'Prevalence': actual.mean(),
    })

per_class = pd.DataFrame(rows).set_index('Class')
per_class.loc['Macro average'] = per_class.mean()
print('Development set | One class vs. the other four | Decision rule: argmax')
display(per_class.round(4))

Development set | One class vs. the other four | Decision rule: argmax


,AUROC,AUPRC (AP),Sensitivity,Specificity,Prevalence
Class,,,,,
BACKGROUND,0.8740,0.3930,0.5274,0.9231,0.1142
OBJECTIVE,0.9243,0.3854,0.0000,1.0000,0.0786
METHODS,0.8805,0.7294,0.7559,0.8478,0.3298
RESULTS,0.8974,0.7540,0.8317,0.8374,0.3257
CONCLUSIONS,0.9588,0.7167,0.7139,0.9627,0.1517
Macro average,0.9070,0.5957,0.5658,0.9142,0.2000


## Uncertainty: bootstrap over whole abstracts
Sentences of the same abstract stay together, as in 03b, so the interval reflects abstract-level variation.

In [7]:
abstract_index, abstract_ids = pd.factorize(dev.abstract_id)
actual_codes = pd.Categorical(dev.label, categories=CLASSES).codes
predicted_codes = pd.Categorical(dev_pred, categories=CLASSES).codes

n_abstracts, n_classes = len(abstract_ids), len(CLASSES)
counts_by_abstract = np.zeros((n_abstracts, n_classes, n_classes), dtype=int)
np.add.at(counts_by_abstract, (abstract_index, actual_codes, predicted_codes), 1)

rng = np.random.default_rng(SEED)
bootstrap_f1 = np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    cm = counts_by_abstract[rng.integers(0, n_abstracts, size=n_abstracts)].sum(axis=0)
    denominator = cm.sum(axis=0) + cm.sum(axis=1)
    class_f1 = np.divide(2 * np.diag(cm), denominator, out=np.zeros(n_classes), where=denominator > 0)
    bootstrap_f1[b] = class_f1.mean()

lower, upper = np.percentile(bootstrap_f1, [2.5, 97.5])
print(f'Dev abstracts: {n_abstracts:,}')
print(f'Dev macro-F1: {macro_f1:.4f}  (95% bootstrap CI {lower:.4f} to {upper:.4f})')

Dev abstracts: 2,500
Dev macro-F1: 0.5477  (95% bootstrap CI 0.5426 to 0.5529)


## Save predictions and results (same format as the other models)

In [8]:
output = dev[['abstract_id', 'sentence_idx', 'label']].copy()
output['prediction'] = dev_pred
for i, c in enumerate(CLASSES):
    output[f'prob_{c}'] = dev_probs[:, i]
output.to_csv(PREDICTIONS_PATH, index=False)

summary = pd.DataFrame([{
    'model': 'Position baseline (10 bins, no text)',
    'split': 'dev', 'sentences': len(dev), 'abstracts': n_abstracts,
    'decision_rule': 'argmax',
    'accuracy': accuracy, 'macro_f1': macro_f1,
    'macro_f1_ci95_lower': lower, 'macro_f1_ci95_upper': upper,
    'macro_auroc': per_class.loc['Macro average', 'AUROC'],
    'macro_average_precision': per_class.loc['Macro average', 'AUPRC (AP)'],
    'fit_seconds': round(fit_seconds, 3), 'hardware': 'CPU',
}])
summary.to_csv(f'{RESULTS_DIR}/dev_{RUN_NAME}_summary.csv', index=False)
per_class.to_csv(f'{RESULTS_DIR}/dev_{RUN_NAME}_per_class.csv')
confusion.to_csv(f'{RESULTS_DIR}/dev_{RUN_NAME}_confusion_matrix.csv')

print('Saved predictions to', PREDICTIONS_PATH)
print('Saved 3 result tables to', RESULTS_DIR)
display(summary.T)

Saved predictions to /content/drive/MyDrive/exsu_project/predictions/dev_position_baseline.csv
Saved 3 result tables to /content/drive/MyDrive/exsu_project/results


,0
model,"Position baseline (10 bins, no text)"
split,dev
sentences,30212
abstracts,2500
decision_rule,argmax
accuracy,0.6887
macro_f1,0.547729
macro_f1_ci95_lower,0.542632
macro_f1_ci95_upper,0.552859
macro_auroc,0.907001
